# Text Prompt Segmentation

This section will show how to use the text prompt segmentation model to segment text prompts from images.

In [ ]:
# Import libraries

import geoai

In [ ]:
# Download raster data

raster_url = (
    "https://huggingface.co/datasets/giswqs/geospatial/resolve/main/trees_brazil.tif"
)
raster_path = geoai.download_file(raster_url)

In [ ]:
# Move the downloaded files to the target folder

import os
import shutil
from pathlib import Path

# ==== INPUTS ====

target_folder = "data/geoai/text-prompt-segmentation"  # Enter your target folder path here
datasets = ["trees_brazil.tif"]  # Add dataset names as a list
source_dir = Path(".")  # Current location of datasets

# ==== SCRIPT ====

target_path = Path(target_folder)
target_path.mkdir(parents=True, exist_ok=True)
for dataset_name in datasets:
    source_path = source_dir / dataset_name
    
    if not source_path.exists():
        print(f"Dataset '{dataset_name}' not found in {source_dir}")
        continue
    
    if source_path.is_file():
        files_to_move = [source_path]
    else:
        files_to_move = list(source_path.parent.glob(f"{dataset_name}.*"))
    
    for file_path in files_to_move:
        dest = target_path / file_path.name
        shutil.move(str(file_path), str(dest))
        print(f"Moved: {file_path.name} -> {target_folder}")
        
print("Done!")

In [ ]:
# Visualize the data

raster = "data/geoai/text-prompt-segmentation/trees_brazil.tif"
geoai.view_raster(raster_url)

In [ ]:
# Initialize the model

segmenter = geoai.CLIPSegmentation(tile_size=512, overlap=32)

In [ ]:
# Run inference on the text prompt

output_path = "data/geoai/text-prompt-segmentation/tree_masks.tif"
text_prompt = "trees"

In [ ]:
# Run the segmentation on the image with the specified text prompt and save the output mask

segmenter.segment_image(
    raster,
    output_path=output_path,
    text_prompt=text_prompt,
    threshold=0.5,
    smoothing_sigma=1.0,
)

In [ ]:
# Visualize the segmentation results 

geoai.view_raster(
    output_path,
    nodata=0,
    opacity=0.8,
    colormap="greens",
    layer_name="Trees",
    basemap=raster_url,
)

In [ ]:
# Create a split map to compare the original image and the segmentation results

geoai.create_split_map(
    left_layer=output_path,
    right_layer=raster_url,
    left_label="Trees",
    right_label="Satellite Image",
    left_args={"nodata": 0, "opacity": 0.8, "colormap": "greens"},
    basemap=raster_url,
)

In [ ]:
# DONE